# Week 5 — `min`, `max`, and `concat`

Each Maddison file in `week-05/materials/maddison_by_country/` is one entity, with columns `Entity`, `Code`, `Year`, and `GDP`.

`.min()` and `.max()` reduce a column to one value. `pd.concat` stacks those files back into one table.

Run this notebook from `introduction-to-data-engineering`, from `week-05/materials`, or from `week-05/student_work`.


## Find the folder


In [ ]:
from pathlib import Path

import pandas as pd


def find_existing(candidates):
    for path in candidates:
        if path.exists():
            return path
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Could not find the Maddison folder. Looked in:\n" + searched
    )


cwd = Path.cwd()
maddison_dir = find_existing(
    [
        cwd / "maddison_by_country",
        cwd / "materials/maddison_by_country",
        cwd / "week-05/materials/maddison_by_country",
        cwd / "../materials/maddison_by_country",
        cwd / "week-03/materials/maddison_by_country",
        cwd / "../../week-03/materials/maddison_by_country",
    ]
)
maddison_dir


## One file: Finland

`.min()` is the smallest value in that column. `.max()` is the largest. `Year.min()` is the earliest year in the file. `Year.max()` is the latest year.


In [ ]:
finland = pd.read_csv(maddison_dir / "Finland.csv")
finland.head(3)


In [ ]:
finland["Year"].min(), finland["Year"].max()


In [ ]:
finland["GDP"].min(), finland["GDP"].max()


The number alone leaves out the year it belongs to. `.idxmin()` is the row label where GDP is smallest. `.loc` reads that row. `.idxmax()` does the same for the largest GDP.


In [ ]:
finland.loc[finland["GDP"].idxmin(), ["Entity", "Year", "GDP"]]


In [ ]:
finland.loc[finland["GDP"].idxmax(), ["Entity", "Year", "GDP"]]


Finland's file runs from 1600 to 2022. The smallest GDP is in 1600, and the largest GDP is in 2022. On this file the two year questions happen to land on the same rows.


## The two year questions on Afghanistan

`Year.min()` answers "what is the earliest year in these rows?".

`df.loc[df["GDP"].idxmin(), "Year"]` answers "in which year is GDP smallest?".

On Afghanistan those years differ. Run both before you decide which question you meant.


In [ ]:
afghanistan = pd.read_csv(maddison_dir / "Afghanistan.csv")
afghanistan["Year"].min(), afghanistan["Year"].max()


In [ ]:
afghanistan.loc[afghanistan["GDP"].idxmin(), ["Entity", "Year", "GDP"]]


In [ ]:
afghanistan.loc[afghanistan["GDP"].idxmax(), ["Entity", "Year", "GDP"]]


The Afghanistan file starts in 1950 and ends in 2022. The smallest GDP is in 1994. The largest GDP is in 2019.


## Stack two files with `concat`

Build a list of DataFrames, then stack them. `ignore_index=True` writes a new row index. The row count equals the sum of the two files.


In [ ]:
finland = pd.read_csv(maddison_dir / "Finland.csv")
sweden = pd.read_csv(maddison_dir / "Sweden.csv")
frames = [finland, sweden]
two_countries = pd.concat(frames, ignore_index=True)
len(finland), len(sweden), len(two_countries)


In [ ]:
len(two_countries) == len(finland) + len(sweden)


In [ ]:
two_countries["Entity"].unique()


## In depth: stack every file

The loop reads each CSV into a list. `concat` stacks the list. One entity per file means `nunique()` matches the number of files.

Compare Finland inside the stacked table with `Finland.csv` on `Entity`, `Year`, and `GDP`.


In [ ]:
all_frames = []
for path in sorted(maddison_dir.glob("*.csv")):
    all_frames.append(pd.read_csv(path))

df = pd.concat(all_frames, ignore_index=True)
len(all_frames), df["Entity"].nunique(), len(df)


In [ ]:
len(df) == sum(len(frame) for frame in all_frames)


In [ ]:
columns = ["Entity", "Year", "GDP"]
from_file = finland[columns].reset_index(drop=True)
from_all = df.loc[df["Entity"] == "Finland", columns].reset_index(drop=True)
from_file.equals(from_all)


## Practice

1. Read `Sweden.csv`. Report the earliest year, the latest year, the smallest GDP, the largest GDP, and the year of the smallest GDP.
2. You already stacked Finland and Sweden. What is `len(two_countries)`, and which two names are in `Entity`?
3. From the full table `df`, show the Afghanistan row where GDP is largest. Keep `Entity`, `Year`, and `GDP`.


In [ ]:
# Practice 1


In [ ]:
# Practice 2


In [ ]:
# Practice 3


## Quiz

**1.** What does `finland["Year"].min()` return?

- A) The whole first row
- B) The earliest year in that column
- C) The year in which GDP is smallest
- D) The number of rows

**2.** Which line returns the year in which GDP is smallest?

- A) `df["Year"].min()`
- B) `df["GDP"].min()`
- C) `df.loc[df["GDP"].idxmin(), "Year"]`
- D) `df["Year"].max()`

**3.** `pd.concat([finland, sweden], ignore_index=True)` produces a table whose row count is:

- A) The larger of the two files
- B) The sum of the two files' row counts
- C) The number of columns
- D) One row per year that appears in both files

**4.** On Afghanistan, `Year.min()` is 1950. The year of the smallest GDP is:

- A) 1950
- B) 1994
- C) 2019
- D) 2022


## Answer key

For the instructor. Remove this section before sharing the notebook if the quiz should be unseen.

**Quiz:** 1 B, 2 C, 3 B, 4 B. The largest Afghanistan GDP is in 2019 (option C is the trap).

**Practice:**

1. Sweden: earliest year 1500, latest year 2022, smallest GDP 956080000 in 1600, largest GDP 494214521158 in 2022.
2. `len(two_countries)` is 373 (167 + 206). The names are Finland and Sweden.
3. Afghanistan, year 2019, GDP 73085701943.

```python
sweden["Year"].min(), sweden["Year"].max()
sweden["GDP"].min(), sweden["GDP"].max()
sweden.loc[sweden["GDP"].idxmin(), ["Entity", "Year", "GDP"]]
```

Filter first, then `idxmax` on that table:

```python
afghanistan_rows = df[df["Entity"] == "Afghanistan"]
afghanistan_rows.loc[
    afghanistan_rows["GDP"].idxmax(), ["Entity", "Year", "GDP"]
]
```
